# 🚦 RoadRisk AI — 01: Raw Dataset Inspection & Schema Exploration

**Objective:** Inspect all 5 project datasets, document their schemas, coordinate reference systems, provenance, labels, missingness, and verify that they are kept cleanly separated without artificial joins.

### Datasets Inspected:
1. **Indian Road Accident Dataset (2022–2025):** 20,000 records across 8 major Indian cities
2. **Road Traffic Accident Severity:** 8,210 train / 4,106 test records with class imbalance
3. **UrbanGB Coordinates:** 360,177 Great Britain coordinate pairs with 469 urban clusters
4. **OpenCity India 2023:** 3,253 highway chainage records with accident severity codes
5. **CCTV Accident Footage Dataset:** 990 image frames categorized into Accident / Non-Accident


In [ ]:
import sys
from pathlib import Path
# Add project root to sys.path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from PIL import Image

from src.config import RAW_DATA_DIR, CCTV_TRAIN_DIR
from src.data_loader import (
    load_indian_roads_dataset,
    load_severity_train_dataset,
    load_severity_test_dataset,
    load_urban_gb_dataset,
    load_opencity_dataset,
    get_cctv_image_manifest
)
from src.data_validation import (
    check_dataframe_coordinates,
    generate_data_quality_report,
    detect_target_leakage_features
)


## 1. Indian Road Accident Dataset (2022–2025)
This dataset provides historical accident records across 8 Indian cities.


In [ ]:
df_indian = load_indian_roads_dataset()
print('Dimensions:', df_indian.shape)
df_indian.head(3)


In [ ]:
# Check geographic coordinate validity in India (bounds: Lat 6-38, Lon 68-98)
geo_report = check_dataframe_coordinates(df_indian, 'latitude', 'longitude', region='india')
print('Geographic Bounds Report:', geo_report)

# Target distribution
print('\nAccident Severity Breakdown:')
print(df_indian['accident_severity'].value_counts())


## 2. Road Traffic Accident Severity (ML Pipeline Dataset)
Features and target classes for the accident severity classification model.


In [ ]:
df_train = load_severity_train_dataset()
df_test = load_severity_test_dataset()
print(f'Train size: {df_train.shape}, Test size: {df_test.shape}')

# Inspect class imbalance in target
print('\nTarget Class Distribution (Accident_severity):')
print(df_train['Accident_severity'].value_counts(normalize=True))

# Check for target leakage features
suspicious = detect_target_leakage_features(df_train.columns.tolist())
print('\nPotential Target Leakage Features Flagged:', suspicious)


## 3. UrbanGB Great Britain Accident Coordinates
**Crucial Distinction:** These coordinates are strictly from Great Britain and must NOT be mixed with Indian road safety data.


In [ ]:
df_ugb = load_urban_gb_dataset(nrows=50000)
print('UrbanGB Points Loaded:', len(df_ugb))
display(df_ugb.head(3))

# Validate that coordinates are in UK bounding box
uk_geo = check_dataframe_coordinates(df_ugb, 'latitude', 'longitude', region='uk')
print('\nUK Coordinates Validation Report:', uk_geo)


## 4. CCTV Accident Detection Imagery
Examine image counts, resolutions, and splits for computer vision modeling.


In [ ]:
manifest = get_cctv_image_manifest()
print('Total Images Indexed:', len(manifest))
print(manifest.groupby(['split', 'class_label']).size())

# Sample first image
if len(manifest) > 0:
    sample_path = manifest.iloc[0]['filepath']
    with Image.open(sample_path) as img:
        print(f'Sample Image Format: {img.format}, Dimensions: {img.size}, Mode: {img.mode}')
